# NIST AI Risk RAG Assistant
## Stage 1 - Environment, loading, inspection, and text-quality validation

This notebook is intentionally built in stages. The current stage establishes a reproducible, page-preserving ingestion boundary before chunking, embedding, retrieval, generation, and evaluation are added.

### Locked design decisions

- **PyPDF** is sufficient because these NIST PDFs contain extractable text; OCR is only flagged if page-level quality checks show it is necessary.
- A page is the provenance boundary. Later, **450-token chunks with 80-token overlap will never cross pages**, so every citation maps to one source page.
- `sentence-transformers/all-MiniLM-L6-v2` will run on **CPU**, preserving the Quadro M1200 VRAM for Ollama.
- ChromaDB will use cosine distance, persist locally, and export to `backend/data/vector_store`. Retrieval will use `top_k=4`.
- Ollama generation will use `qwen3:4b`, a 4096-token context, and temperature 0.1. Thinking remains enabled and will later be returned separately from the final answer.
- Every chunk will carry `document_id`, document title, one-based PDF page, source URL, and deterministic chunk ID.

## 1. Environment setup

This cell records the runtime instead of mutating it. Install dependencies in the project virtual environment before starting Jupyter; package installation inside a notebook makes `Restart and Run All` less deterministic.

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install jupyter pandas numpy chromadb sentence-transformers pypdf ollama
python -m ipykernel install --user --name nist-rag --display-name 'Python (nist-rag)'
```

In [6]:
%pip install jupyter pandas numpy chromadb sentence-transformers pypdf ollama

  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached scipy-1.18.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached

In [13]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import re
import subprocess
import sys
import unicodedata
from collections import Counter
from dataclasses import asdict, dataclass
from importlib import metadata as importlib_metadata
from pathlib import Path
from typing import Any, Iterable, Sequence

import pandas as pd
from pypdf import PdfReader

try:
    from IPython.display import display
except ImportError:  # Keeps validation possible in a plain Python process.
    def display(value: object) -> None:
        print(value)

assert sys.version_info >= (3, 10), "Python 3.10+ is required."
pd.set_option("display.max_colwidth", 120)

In [14]:
@dataclass(frozen=True, slots=True)
class PipelineConfig:
    """Immutable configuration shared by notebook and backend stages."""

    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    embedding_device: str = "cpu"
    chunk_size_tokens: int = 450
    chunk_overlap_tokens: int = 80
    collection_name: str = "nist_ai_risk_corpus"
    distance_metric: str = "cosine"
    top_k: int = 4
    ollama_model: str = "qwen3:4b"
    ollama_context_tokens: int = 4096
    temperature: float = 0.1
    expected_documents: int = 3
    expected_pages: int = 259
    expected_evaluation_questions: int = 12

CONFIG = PipelineConfig()
CONFIG

PipelineConfig(embedding_model='sentence-transformers/all-MiniLM-L6-v2', embedding_device='cpu', chunk_size_tokens=450, chunk_overlap_tokens=80, collection_name='nist_ai_risk_corpus', distance_metric='cosine', top_k=4, ollama_model='qwen3:4b', ollama_context_tokens=4096, temperature=0.1, expected_documents=3, expected_pages=259, expected_evaluation_questions=12)

In [15]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root independently of the Jupyter launch directory."""
    origin = (start or Path.cwd()).resolve()
    candidates = [origin, *origin.parents, origin / "rag-assistant-project"]
    for candidate in candidates:
        if (candidate / "data" / "raw").is_dir() and (candidate / "data" / "metadata" / "sources.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Expected data/raw and data/metadata/sources.json."
    )


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
SOURCE_MANIFEST_PATH = PROJECT_ROOT / "data" / "metadata" / "sources.json"
EVALUATION_PATH = PROJECT_ROOT / "data" / "evaluation_questions.csv"
VECTOR_STORE_EXPORT_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {platform.python_version()} ({platform.system()} {platform.machine()})")

Project root: /home/gamal/Projects/RAG-Powered-Document-Assistant
Python: 3.14.7 (Linux x86_64)


In [16]:
def installed_version(distribution: str) -> str:
    """Return an installed distribution version without importing heavy packages."""
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return "NOT INSTALLED"


def command_output(command: Sequence[str], timeout_seconds: int = 8) -> str:
    """Run a read-only environment check and return a stable status string."""
    try:
        result = subprocess.run(
            list(command), capture_output=True, text=True, check=False, timeout=timeout_seconds
        )
    except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
        return f"unavailable: {type(exc).__name__}"
    output = (result.stdout or result.stderr).strip()
    return output if output else f"exit_code={result.returncode}"


packages = ["pypdf", "pandas", "numpy", "chromadb", "sentence-transformers", "ollama", "jupyter"]
environment_report = pd.DataFrame(
    {"component": packages, "version": [installed_version(name) for name in packages]}
)
display(environment_report)
print("Ollama:", command_output(["ollama", "--version"]))
print("Loaded Ollama models:\n", command_output(["ollama", "ps"]))

,component,version
0,pypdf,6.18.1
1,pandas,3.0.5
2,numpy,2.5.3
3,chromadb,1.5.9
4,sentence-transformers,6.0.1
5,ollama,0.6.2
6,jupyter,1.1.1


Ollama: ollama version is 0.34.0
Loaded Ollama models:
 NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL


## 2. Load and validate source metadata

The manifest is authoritative for identity and provenance. SHA-256 and page-count checks prevent silent corpus drift, while the evaluation file is validated but never ingested into the vector store.

In [18]:
@dataclass(frozen=True, slots=True)
class SourceDocument:
    """Validated source metadata for one PDF."""

    document_id: str
    file_name: str
    title: str
    source_url: str
    sha256: str
    expected_pages: int

    @property
    def path(self) -> Path:
        return RAW_DATA_DIR / self.file_name


def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    """Calculate a file digest using bounded memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def load_source_manifest(path: Path) -> list[SourceDocument]:
    """Load the source manifest and reject incomplete or duplicate records."""
    payload = json.loads(path.read_text(encoding="utf-8"))
    raw_sources = payload.get("sources", [])
    if not raw_sources:
        raise ValueError(f"No sources found in {path}")
    sources = [SourceDocument(**record) for record in raw_sources]
    document_ids = [source.document_id for source in sources]
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("document_id values must be unique.")
    return sources


def validate_source_files(sources: Sequence[SourceDocument]) -> pd.DataFrame:
    """Validate presence, checksums, and physical page counts before extraction."""
    rows: list[dict[str, Any]] = []
    for source in sources:
        exists = source.path.is_file()
        actual_sha256 = sha256_file(source.path) if exists else None
        actual_pages = len(PdfReader(source.path).pages) if exists else None
        rows.append(
            {
                "document_id": source.document_id,
                "file_name": source.file_name,
                "exists": exists,
                "checksum_ok": actual_sha256 == source.sha256,
                "expected_pages": source.expected_pages,
                "actual_pages": actual_pages,
                "page_count_ok": actual_pages == source.expected_pages,
            }
        )
    return pd.DataFrame(rows)


sources = load_source_manifest(SOURCE_MANIFEST_PATH)
source_validation = validate_source_files(sources)
display(source_validation)

evaluation_questions = pd.read_csv(EVALUATION_PATH)
required_evaluation_columns = {
    "question_id", "question", "expected_document_id", "expected_answer_points", "answerable"
}
missing_columns = required_evaluation_columns - set(evaluation_questions.columns)
assert not missing_columns, f"Missing evaluation columns: {sorted(missing_columns)}"
assert len(evaluation_questions) == CONFIG.expected_evaluation_questions
assert evaluation_questions["question_id"].is_unique
print(f"Validated {len(evaluation_questions)} evaluation questions; they will not be embedded.")

,document_id,file_name,exists,checksum_ok,expected_pages,actual_pages,page_count_ok
0,nist_ai_rmf_1_0,nist_ai_rmf_1_0.pdf,True,True,48,48,True
1,nist_genai_profile,nist_genai_profile.pdf,True,True,64,64,True
2,nist_ai_rmf_playbook,nist_ai_rmf_playbook.pdf,True,True,147,147,True


Validated 12 evaluation questions; they will not be embedded.


## 3. Page-preserving PDF extraction

Each output record represents exactly one physical PDF page. Text normalization removes Unicode compatibility variants, soft hyphens, and unstable horizontal whitespace, but preserves line boundaries and list structure needed for later inspection and chunking. Extraction errors are captured per page and then enforced by a quality gate.

In [19]:
@dataclass(frozen=True, slots=True)
class PageRecord:
    """Text and provenance for one physical PDF page."""

    document_id: str
    title: str
    page: int
    source_url: str
    text: str
    extraction_error: str | None = None


def normalize_extracted_text(text: str) -> str:
    """Normalize extraction noise without collapsing meaningful line boundaries."""
    normalized = unicodedata.normalize("NFKC", text).replace("\u00ad", "").replace("\u00a0", " ")
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in normalized.splitlines()]
    return "\n".join(line for line in lines if line)


def extract_pdf_pages(source: SourceDocument) -> list[PageRecord]:
    """Extract one record per page and retain failures for deterministic QA."""
    reader = PdfReader(source.path)
    records: list[PageRecord] = []
    for page_number, pdf_page in enumerate(reader.pages, start=1):
        try:
            text = normalize_extracted_text(pdf_page.extract_text() or "")
            error = None
        except Exception as exc:  # Preserve the failed page and surface it in the gate.
            text = ""
            error = f"{type(exc).__name__}: {exc}"
        records.append(
            PageRecord(
                document_id=source.document_id,
                title=source.title,
                page=page_number,
                source_url=source.source_url,
                text=text,
                extraction_error=error,
            )
        )
    return records


def extract_corpus(sources: Sequence[SourceDocument]) -> pd.DataFrame:
    """Extract the small corpus sequentially to keep memory and failure semantics simple."""
    records = [record for source in sources for record in extract_pdf_pages(source)]
    frame = pd.DataFrame(asdict(record) for record in records)
    return frame.sort_values(["document_id", "page"], kind="stable").reset_index(drop=True)


pages_df = extract_corpus(sources)
print(f"Extracted {len(pages_df):,} page records from {pages_df['document_id'].nunique()} PDFs.")

Extracted 259 page records from 3 PDFs.


## 4. Inspection and text-quality validation

Sparse title or section-divider pages are warnings, not automatic OCR failures. A document is marked as needing OCR only when extraction fails or more than 20% of its pages contain fewer than 40 characters. Character checks also surface control characters, replacement glyphs, and unusually low alphabetic content.

In [20]:
def page_quality_issues(text: str, extraction_error: str | None) -> list[str]:
    """Return non-exclusive quality findings for a single extracted page."""
    if extraction_error:
        return ["extraction_error"]
    if not text.strip():
        return ["blank"]

    issues: list[str] = []
    non_space = [char for char in text if not char.isspace()]
    printable_ratio = sum(char.isprintable() for char in text) / max(len(text), 1)
    alphabetic_ratio = sum(char.isalpha() for char in non_space) / max(len(non_space), 1)
    control_count = sum(ord(char) < 32 and char not in "\n\r\t" for char in text)

    if len(text) < 40:
        issues.append("near_empty")
    if len(text) >= 40 and alphabetic_ratio < 0.35:
        issues.append("low_alphabetic_ratio")
    if printable_ratio < 0.98:
        issues.append("low_printable_ratio")
    if "\ufffd" in text:
        issues.append("replacement_character")
    if control_count:
        issues.append("control_characters")
    return issues


def repeated_edge_lines(document_pages: pd.DataFrame, edge_depth: int = 2) -> list[tuple[str, int]]:
    """Find probable repeated headers/footers for later conservative removal."""
    counts: Counter[str] = Counter()
    for text in document_pages["text"]:
        lines = [line.strip() for line in str(text).splitlines() if line.strip()]
        edge_lines = lines[:edge_depth] + lines[-edge_depth:]
        normalized = {re.sub(r"\d+", "#", line).casefold() for line in edge_lines if len(line) >= 5}
        counts.update(normalized)
    minimum_count = max(3, math.ceil(len(document_pages) * 0.30))
    return sorted(
        [(line, count) for line, count in counts.items() if count >= minimum_count],
        key=lambda item: (-item[1], item[0]),
    )


pages_df["char_count"] = pages_df["text"].str.len()
pages_df["word_count"] = pages_df["text"].str.findall(r"\b\w+\b").str.len()
pages_df["quality_issues"] = [
    page_quality_issues(text, error)
    for text, error in zip(pages_df["text"], pages_df["extraction_error"], strict=True)
]
pages_df["has_quality_warning"] = pages_df["quality_issues"].str.len().gt(0)

summary_rows: list[dict[str, Any]] = []
repeated_lines_by_document: dict[str, list[tuple[str, int]]] = {}
for document_id, group in pages_df.groupby("document_id", sort=False):
    sparse_pages = group.loc[group["char_count"] < 40, "page"].astype(int).tolist()
    extraction_failures = int(group["extraction_error"].notna().sum())
    ocr_required = extraction_failures > 0 or len(sparse_pages) / len(group) > 0.20
    repeated_lines_by_document[document_id] = repeated_edge_lines(group)
    summary_rows.append(
        {
            "document_id": document_id,
            "pages": len(group),
            "words": int(group["word_count"].sum()),
            "median_chars_per_page": int(group["char_count"].median()),
            "sparse_pages": sparse_pages,
            "warning_pages": int(group["has_quality_warning"].sum()),
            "extraction_failures": extraction_failures,
            "ocr_required": ocr_required,
        }
    )

quality_summary = pd.DataFrame(summary_rows)
display(quality_summary)
print("Probable repeated page-edge lines (candidates for later cleaning):")
for document_id, candidates in repeated_lines_by_document.items():
    print(f"- {document_id}: {candidates or 'none detected'}")

,document_id,pages,words,median_chars_per_page,sparse_pages,warning_pages,extraction_failures,ocr_required
0,nist_ai_rmf_1_0,48,16261,2446,[],10,0,False
1,nist_ai_rmf_playbook,147,45172,2290,"[1, 5, 37, 61, 97]",8,0,False
2,nist_genai_profile,64,23105,2572,[],10,0,False


Probable repeated page-edge lines (candidates for later cleaning):
- nist_ai_rmf_1_0: [('nist ai #-# ai rmf #.#', 43), ('page #', 42)]
- nist_ai_rmf_playbook: [('# of #', 139)]
- nist_genai_profile: none detected


In [21]:
def preview_pages(frame: pd.DataFrame, preview_chars: int = 300) -> pd.DataFrame:
    """Return first, middle, and last page previews for each document."""
    selections: list[pd.DataFrame] = []
    for _, group in frame.groupby("document_id", sort=False):
        positions = sorted({0, len(group) // 2, len(group) - 1})
        selections.append(group.iloc[positions])
    sample = pd.concat(selections, ignore_index=True).copy()
    sample["text_preview"] = sample["text"].str.replace("\n", " ", regex=False).str.slice(0, preview_chars)
    return sample[["document_id", "page", "char_count", "quality_issues", "text_preview"]]


display(preview_pages(pages_df))
warning_pages = pages_df.loc[
    pages_df["has_quality_warning"],
    ["document_id", "page", "char_count", "quality_issues", "text"],
].copy()
warning_pages["text_preview"] = warning_pages.pop("text").str.replace("\n", " ", regex=False).str.slice(0, 180)
display(warning_pages)

,document_id,page,char_count,quality_issues,text_preview
0,nist_ai_rmf_1_0,1,76,[low_printable_ratio],NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)
1,nist_ai_rmf_1_0,25,1289,[],NIST AI 100-1 AI RMF 1.0 Part 2: Core and Profiles 5. AI RMF Core The AI RMF Core provides outcomes and actions that...
2,nist_ai_rmf_1_0,48,88,[],This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1
3,nist_ai_rmf_playbook,1,29,"[near_empty, low_printable_ratio]",AI RMFAI RMF PLAYBOOKPLAYBOOK
4,nist_ai_rmf_playbook,74,2306,[],"70 of 142 Andrew D. Selbst, danah boyd, Sorelle A. Friedler, et al. 2019. Fairness and Abstraction in Sociotechnical..."
5,nist_ai_rmf_playbook,147,2203,[],"142 of 142 George Margetis, Stavroula Ntoa, Margherita Antona, and Constantine Stephanidis. “Human- Centered Design ..."
6,nist_genai_profile,1,232,[low_printable_ratio],NIST Trustworthy and Responsible AI NIST AI 600-1 Artificial Intelligence Risk Management Framework: Generative Arti...
7,nist_genai_profile,33,2730,[],29 MS-1.1-006 Implement continuous monitoring of GAI system impacts to identify whether GAI outputs are equitable ac...
8,nist_genai_profile,64,876,[],"60 Zhang, Y . et al. (2023) Human favoritism, not AI aversion: People’s perceptions (and bias) toward generative AI,..."


,document_id,page,char_count,quality_issues,text_preview
0,nist_ai_rmf_1_0,1,76,[low_printable_ratio],NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)
1,nist_ai_rmf_1_0,2,376,[low_printable_ratio],NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0) This publication is available free of c...
3,nist_ai_rmf_1_0,4,1235,[low_printable_ratio],Table of Contents Executive Summary 1 Part 1: Foundational Information 4 1 Framing Risk 4 1.1 Understanding and Addr...
27,nist_ai_rmf_1_0,28,2309,[low_printable_ratio],NIST AI 100-1 AI RMF 1.0 Table 1: Categories and subcategories for the GOVERN function. (Continued) GOVERN 1.5: Ongo...
28,nist_ai_rmf_1_0,29,2470,[low_printable_ratio],NIST AI 100-1 AI RMF 1.0 Table 1: Categories and subcategories for the GOVERN function. (Continued) that considers a...
30,nist_ai_rmf_1_0,31,2094,[low_printable_ratio],NIST AI 100-1 AI RMF 1.0 Table 2: Categories and subcategories for the MAP function. MAP 1: Context is established a...
31,nist_ai_rmf_1_0,32,2334,[low_printable_ratio],NIST AI 100-1 AI RMF 1.0 Table 2: Categories and subcategories for the MAP function. (Continued) MAP 2.3: Scientific...
33,nist_ai_rmf_1_0,34,2103,[low_printable_ratio],NIST AI 100-1 AI RMF 1.0 Practices related to measuring AI risks are described in the NIST AI RMF Playbook. Table 3 ...
34,nist_ai_rmf_1_0,35,2063,[low_printable_ratio],NIST AI 100-1 AI RMF 1.0 Table 3: Categories and subcategories for the MEASURE function. (Continued) MEASURE 2.6: Th...
36,nist_ai_rmf_1_0,37,2265,[low_printable_ratio],NIST AI 100-1 AI RMF 1.0 Practices related to managing AI risks are described in the NIST AI RMF Playbook. Table 4 l...


In [22]:
def enforce_ingestion_gate(
    source_validation: pd.DataFrame,
    pages: pd.DataFrame,
    quality_summary: pd.DataFrame,
    config: PipelineConfig,
) -> None:
    """Fail early if corpus identity, completeness, or extraction is unsafe."""
    assert len(source_validation) == config.expected_documents, "Unexpected document count."
    assert source_validation["exists"].all(), "One or more PDFs are missing."
    assert source_validation["checksum_ok"].all(), "A PDF checksum differs from sources.json."
    assert source_validation["page_count_ok"].all(), "A PDF page count differs from sources.json."
    assert len(pages) == config.expected_pages, "Unexpected total page count."
    assert pages[["document_id", "page"]].duplicated().sum() == 0, "Duplicate page identity found."
    assert quality_summary["extraction_failures"].eq(0).all(), "At least one page failed extraction."
    assert not quality_summary["ocr_required"].any(), "OCR is required before ingestion."


enforce_ingestion_gate(source_validation, pages_df, quality_summary, CONFIG)
print("PASS: corpus identity, page completeness, and extraction quality are safe for chunking.")

PASS: corpus identity, page completeness, and extraction quality are safe for chunking.


### Stage 1 result

The corpus contains **3 text PDFs and 259 page records**. No files failed to parse and OCR is not required. Sparse Playbook divider pages remain in the page table as transparent warnings; they should not produce chunks unless they contain enough semantic content after conservative header/footer cleaning.

**Next stage:** implement deterministic repeated-margin cleanup, tokenizer-aware 450/80 page-bounded chunking, stable chunk IDs, and chunk-level validation. Embeddings, Chroma persistence, Ollama prompting with separate thinking, and all-12-question evaluation will follow after the chunk contract is verified.